# 03 - RQ1: Which pre-departure characteristics influence late arrival?

**Outcome (one DV):** `is_late` = `ArrDel15` (1 = arrived 15+ minutes after schedule), binary.

**Design**
* **Primary model - logistic regression (statsmodels).** It gives interpretable coefficients and formal tests,
  which is what "which characteristics influence" needs. H0: every coefficient = 0 (likelihood-ratio test);
  each factor is also tested on its own (drop-one likelihood-ratio test).
* **Comparison models - random forest and XGBoost**, to check whether the same factors matter when
  non-linear effects and interactions are allowed. **SHAP** ranks factor influence on the XGBoost model.
* **Baseline - majority class** ("always on time").
* **Validation - 80/20 stratified hold-out split + 3-fold CV** on a subsample.

**Leakage control.** Only information known *before departure* is used: month, weekday, **scheduled**
departure block, distance, carrier, origin, destination and the *previous month's* route load factor.
Excluded on purpose: `DepDelay`, `DepDel15`, taxi times, wheels-off/on, elapsed time actuals and every
delay-cause field - all are recorded during or after the flight and would leak the answer.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from config import *
from results_store import save_result, load_results
from labels import MONTH_NAMES, DOW_NAMES, carrier_name, airport_name
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score
import xgboost as xgb
from metrics import classification_report_dict, confusion_counts
from plot_style import set_style, save_fig
set_style()

## 1. Feature engineering

In [2]:
df = pd.read_csv(PROCESSED / "rq1_sample.csv.gz")
df = df.dropna(subset=["is_late", "distance", "carrier", "origin", "dest", "dep_time_blk"]).reset_index(drop=True)
y = df["is_late"].astype(int).values
print(f"{len(df):,} flights | late rate {y.mean()*100:.2f}%")

# Group rare airports (keeps the design matrix small and every level well populated)
top_o = df["origin"].value_counts().head(TOP_AIRPORTS_RQ1).index
top_d = df["dest"].value_counts().head(TOP_AIRPORTS_RQ1).index
df["origin_g"] = df["origin"].where(df["origin"].isin(top_o), "Other")
df["dest_g"] = df["dest"].where(df["dest"].isin(top_d), "Other")
df["month_c"] = df["month"].astype(int).astype(str).str.zfill(2)
df["dow_c"] = df["day_of_week"].astype(int).astype(str)
df["distance_k"] = df["distance"] / 1000.0                       # per 1,000 miles
lf_med = df["lf_prev"].median()
df["lf_prev_missing"] = df["lf_prev"].isna().astype(int)         # missing-indicator keeps flights without a prior-month match
df["lf_prev_10"] = (df["lf_prev"].fillna(lf_med) - lf_med) / 10  # per +10 percentage points, centred at the median

def dummies(series, prefix):
    s = series.astype(str)
    ref = s.value_counts().index[0]                               # reference = most common level
    d = pd.get_dummies(s, prefix=prefix, dtype="float32")
    return d.drop(columns=f"{prefix}_{ref}"), ref

spec = {"Month": ("month_c", "month"), "Day of week": ("dow_c", "dow"), "Scheduled departure block": ("dep_time_blk", "blk"),
        "Carrier": ("carrier", "carr"), "Origin airport": ("origin_g", "orig"), "Destination airport": ("dest_g", "dest")}
blocks, refs = {}, {}
for name, (col, pre) in spec.items():
    blocks[name], refs[name] = dummies(df[col], pre)
blocks["Route distance"] = df[["distance_k"]].astype("float32")
blocks["Prior-month load factor"] = df[["lf_prev_10", "lf_prev_missing"]].astype("float32")
X = pd.concat(list(blocks.values()), axis=1)
X = X.loc[:, X.std() > 0]                                         # drop constant columns, if any
col_to_factor = {c: n for n, b in blocks.items() for c in b.columns if c in X.columns}
print("design matrix:", X.shape, "| reference levels:", refs)

400,000 flights | late rate 21.58%


design matrix: (400000, 102) | reference levels: {'Month': '07', 'Day of week': '1', 'Scheduled departure block': '0700-0759', 'Carrier': 'WN', 'Origin airport': 'Other', 'Destination airport': 'Other'}


In [3]:
# Feature table used in the report (rationale + which models use it)
feat = pd.DataFrame([
    ["Month", "Original", "Categorical (12)", "Seasonal weather and demand cycles", "LR, RF, XGB"],
    ["Day of week", "Original", "Categorical (7)", "Weekly demand and schedule patterns", "LR, RF, XGB"],
    ["Scheduled departure block", "Original", "Categorical (hourly)", "Delays accumulate through the day (knock-on effects)", "LR, RF, XGB"],
    ["Carrier", "Original", "Categorical", "Operating practices differ by airline", "LR, RF, XGB"],
    ["Origin airport", "Engineered (top %d + Other)" % TOP_AIRPORTS_RQ1, "Categorical", "Congestion and weather exposure differ by airport; grouping avoids sparse levels", "LR, RF, XGB"],
    ["Destination airport", "Engineered (top %d + Other)" % TOP_AIRPORTS_RQ1, "Categorical", "Same, for arrival airport", "LR, RF, XGB"],
    ["Route distance (per 1,000 mi)", "Original (rescaled)", "Continuous", "Longer flights have more schedule slack", "LR, RF, XGB"],
    ["Prior-month route load factor (per +10 pts)", "Engineered (T-100 join, previous month)", "Continuous + missing flag", "Fuller aircraft take longer to board; previous month is known before departure", "LR, RF, XGB"],
], columns=["Feature", "Original / engineered", "Type", "Reason for inclusion", "Used in model(s)"])
feat.to_csv(TAB / "rq1_feature_set.csv", index=False)
feat

,Feature,Original / engineered,Type,Reason for inclusion,Used in model(s)
0,Month,Original,Categorical (12),Seasonal weather and demand cycles,"LR, RF, XGB"
1,Day of week,Original,Categorical (7),Weekly demand and schedule patterns,"LR, RF, XGB"
2,Scheduled departure block,Original,Categorical (hourly),Delays accumulate through the day (knock-on ef...,"LR, RF, XGB"
3,Carrier,Original,Categorical,Operating practices differ by airline,"LR, RF, XGB"
4,Origin airport,Engineered (top 25 + Other),Categorical,Congestion and weather exposure differ by airp...,"LR, RF, XGB"
5,Destination airport,Engineered (top 25 + Other),Categorical,"Same, for arrival airport","LR, RF, XGB"
6,"Route distance (per 1,000 mi)",Original (rescaled),Continuous,Longer flights have more schedule slack,"LR, RF, XGB"
7,Prior-month route load factor (per +10 pts),"Engineered (T-100 join, previous month)",Continuous + missing flag,Fuller aircraft take longer to board; previous...,"LR, RF, XGB"


## 2. Train / test split and baseline

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
prev = float(y_tr.mean())          # decision threshold for every model = training prevalence (class imbalance)
print(f"train {len(X_tr):,} | test {len(X_te):,} | prevalence {prev:.4f}")

results = {}
results["Majority-class baseline"] = {"accuracy": float(1 - y_te.mean()), "precision": 0.0, "recall": 0.0, "f1": 0.0, "auc": 0.5}

train 320,000 | test 80,000 | prevalence 0.2158


## 3. Logistic regression (primary model)
Wald z-tests give a p-value per coefficient; the overall likelihood-ratio test compares the full model with an
intercept-only model (H0: all coefficients = 0); a drop-one likelihood-ratio test gives one p-value per factor.

In [5]:
Xc_tr = sm.add_constant(X_tr.astype("float64"))
Xc_te = sm.add_constant(X_te.astype("float64"), has_constant="add")
try:
    logit = sm.Logit(y_tr, Xc_tr).fit(method="newton", maxiter=100, disp=0)
except Exception as e:                       # numerical trouble -> fall back to a more forgiving optimiser
    print("newton failed:", e); logit = sm.Logit(y_tr, Xc_tr).fit(method="bfgs", maxiter=500, disp=0)
converged = bool(logit.mle_retvals.get("converged", True))
p_lr = np.asarray(logit.predict(Xc_te))
results["Logistic regression"] = classification_report_dict(y_te, p_lr, prev)
conf_lr = confusion_counts(y_te, p_lr, prev)      # used for the worked metric example in the report
overall = {"llr_chi2": float(logit.llr), "df": int(logit.df_model), "p_value": float(logit.llr_pvalue),
           "mcfadden_r2": float(logit.prsquared), "converged": converged, "n_train": int(len(y_tr))}
print(overall)

coef = pd.DataFrame({"coef": logit.params, "se": logit.bse, "z": logit.tvalues, "p": logit.pvalues})
ci = logit.conf_int(); coef["OR"] = np.exp(coef["coef"]); coef["OR_lo"] = np.exp(ci[0]); coef["OR_hi"] = np.exp(ci[1])
coef["factor"] = [col_to_factor.get(i, "Intercept") for i in coef.index]
coef.to_csv(TAB / "rq1_logit_coefficients.csv")
coef.drop(index="const").sort_values("z", key=abs, ascending=False).head(12).round(4)

{'llr_chi2': 17898.392695639282, 'df': 102, 'p_value': 0.0, 'mcfadden_r2': 0.0536198818014445, 'converged': True, 'n_train': 320000}


,coef,se,z,p,OR,OR_lo,OR_hi,factor
blk_1800-1859,1.1939,0.0257,46.5127,0.0,3.2998,3.1379,3.4701,Scheduled departure block
blk_1900-1959,1.2065,0.0263,45.9210,0.0,3.3416,3.1739,3.5182,Scheduled departure block
blk_1700-1759,1.1509,0.0257,44.7761,0.0,3.1611,3.0058,3.3244,Scheduled departure block
blk_2000-2059,1.1535,0.0276,41.8277,0.0,3.1691,3.0024,3.3451,Scheduled departure block
blk_1600-1659,1.0493,0.0263,39.9075,0.0,2.8558,2.7123,3.0068,Scheduled departure block
blk_1500-1559,1.0050,0.0264,38.0083,0.0,2.7319,2.5939,2.8772,Scheduled departure block
month_09,-0.8100,0.0217,-37.2859,0.0,0.4449,0.4263,0.4642,Month
month_10,-0.7604,0.0213,-35.7248,0.0,0.4675,0.4484,0.4874,Month
blk_2100-2159,1.0672,0.0301,35.4857,0.0,2.9071,2.7407,3.0836,Scheduled departure block
month_11,-0.7254,0.0215,-33.7524,0.0,0.4841,0.4641,0.5049,Month


In [6]:
# drop-one likelihood-ratio test for each factor (H0: all coefficients of the factor = 0)
rows = []
for name in blocks:
    cols = [c for c in X.columns if col_to_factor.get(c) == name]
    keep = [c for c in Xc_tr.columns if c not in cols]
    red = sm.Logit(y_tr, Xc_tr[keep]).fit(method="newton", maxiter=100, disp=0)
    lr = 2 * (logit.llf - red.llf); dfree = len(cols)
    rows.append({"Factor": name, "df": dfree, "LR_chi2": lr, "p_value": float(stats.chi2.sf(lr, dfree))})
ftest = pd.DataFrame(rows).sort_values("LR_chi2", ascending=False)
ftest["significant_at_.05"] = ftest["p_value"] < 0.05
ftest.to_csv(TAB / "rq1_factor_tests.csv", index=False)
ftest.round(4)

,Factor,df,LR_chi2,p_value,significant_at_.05
2,Scheduled departure block,18,10567.9322,0.0000,True
0,Month,11,3163.8682,0.0000,True
3,Carrier,14,1119.7027,0.0000,True
5,Destination airport,25,896.4313,0.0000,True
1,Day of week,6,799.9172,0.0000,True
4,Origin airport,25,734.0544,0.0000,True
7,Prior-month load factor,2,53.5841,0.0000,True
6,Route distance,1,1.7152,0.1903,False


## 4. Comparison models (random forest, XGBoost)

In [7]:
rf = RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=100, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_tr, y_tr); p_rf = rf.predict_proba(X_te)[:, 1]
results["Random forest"] = classification_report_dict(y_te, p_rf, prev)

xg = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                       tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE, eval_metric="auc")
xg.fit(X_tr, y_tr); p_xg = xg.predict_proba(X_te)[:, 1]
results["XGBoost"] = classification_report_dict(y_te, p_xg, prev)

# 3-fold cross-validated AUC on a subsample (stability check; same folds for all models)
n_sub = min(90_000, len(X_tr)); idx = np.random.RandomState(RANDOM_STATE).choice(len(X_tr), n_sub, replace=False)
Xs, ys = X_tr.iloc[idx], y_tr[idx]
skf = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)
cv = {
    "Logistic regression": cross_val_score(LogisticRegression(C=1e6, max_iter=500), Xs, ys, cv=skf, scoring="roc_auc"),
    "Random forest": cross_val_score(RandomForestClassifier(n_estimators=60, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=RANDOM_STATE), Xs, ys, cv=skf, scoring="roc_auc"),
    "XGBoost": cross_val_score(xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE), Xs, ys, cv=skf, scoring="roc_auc"),
}
for k, v in cv.items():
    results[k]["cv_auc_mean"], results[k]["cv_auc_sd"] = float(v.mean()), float(v.std())

comp = pd.DataFrame(results).T
comp.index.name = "Model"
comp.to_csv(TAB / "rq1_model_comparison.csv")
comp.round(4)

,accuracy,precision,recall,f1,auc,cv_auc_mean,cv_auc_sd
Model,,,,,,,
Majority-class baseline,0.7842,0.0000,0.0000,0.0000,0.5000,NaN,NaN
Logistic regression,0.6032,0.3033,0.6466,0.4129,0.6647,0.6624,0.0028
Random forest,0.5892,0.2937,0.6433,0.4033,0.6515,0.6463,0.0025
XGBoost,0.6237,0.3161,0.6388,0.4229,0.6775,0.6597,0.0017


## 4b. Temporal validation: fit on the earlier year, test on the later year
A random split can be optimistic because flights from the same days sit on both sides. Here every model is fitted only on the
earlier year and scored on the later year, which is how the model would be used in practice.

In [8]:
yrs = sorted(df["year"].astype(int).unique())
temporal = None
if len(yrs) >= 2:
    tr_m = (df["year"].astype(int) == yrs[0]).values; te_m = (df["year"].astype(int) == yrs[-1]).values
    Xa, Xb, ya, yb = X.loc[tr_m], X.loc[te_m], y[tr_m], y[te_m]
    prev_a = float(ya.mean())
    t_res = {"Majority-class baseline": {"accuracy": float(1 - yb.mean()), "precision": 0.0, "recall": 0.0, "f1": 0.0, "auc": 0.5}}
    lr_t = LogisticRegression(C=1e6, max_iter=1000).fit(Xa, ya)
    t_res["Logistic regression"] = classification_report_dict(yb, lr_t.predict_proba(Xb)[:, 1], prev_a)
    rf_t = RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=100, n_jobs=-1, random_state=RANDOM_STATE).fit(Xa, ya)
    t_res["Random forest"] = classification_report_dict(yb, rf_t.predict_proba(Xb)[:, 1], prev_a)
    xg_t = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                             tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE, eval_metric="auc").fit(Xa, ya)
    t_res["XGBoost"] = classification_report_dict(yb, xg_t.predict_proba(Xb)[:, 1], prev_a)
    temporal = {"train_year": int(yrs[0]), "test_year": int(yrs[-1]), "n_train": int(len(ya)), "n_test": int(len(yb)),
                "prevalence_train": prev_a, "prevalence_test": float(yb.mean()), "models": t_res}
    tv = pd.DataFrame(t_res).T; tv.index.name = "Model"; tv.to_csv(TAB / "rq1_temporal_validation.csv")
    print(f"train {yrs[0]} ({len(ya):,} flights) -> test {yrs[-1]} ({len(yb):,} flights)")
    display(tv.round(4))
else:
    print("only one year loaded - temporal validation skipped")

train 2024 (201,303 flights) -> test 2025 (198,697 flights)


,accuracy,precision,recall,f1,auc
Model,,,,,
Majority-class baseline,0.7773,0.0000,0.0000,0.0000,0.5000
Logistic regression,0.6038,0.3026,0.5971,0.4016,0.6427
Random forest,0.5828,0.2913,0.6096,0.3942,0.6300
XGBoost,0.6150,0.3070,0.5795,0.4013,0.6429


## 5. Influence ranking with SHAP (XGBoost) and figures

In [9]:
try:
    import shap
    bg = X_te.sample(min(3000, len(X_te)), random_state=RANDOM_STATE)
    sv = shap.TreeExplainer(xg).shap_values(bg)
    sv = np.asarray(sv[1] if isinstance(sv, list) else sv)
    grp = pd.DataFrame(sv, columns=X.columns).T.groupby(pd.Series(col_to_factor)).sum().T   # factor-level SHAP per flight
    shap_imp = grp.abs().mean().sort_values(ascending=False)
    shap_method = "SHAP (mean |factor contribution|, log-odds)"
except Exception as e:                                                                       # fallback keeps the notebook runnable
    print("SHAP failed, using XGBoost gain importance instead:", e)
    gain = pd.Series(xg.feature_importances_, index=X.columns)
    shap_imp = gain.groupby(pd.Series(col_to_factor)).sum().sort_values(ascending=False)
    shap_method = "XGBoost gain importance"
shap_imp.rename("importance").to_csv(TAB / "rq1_factor_importance.csv")
print(shap_method); shap_imp.round(4)

SHAP (mean |factor contribution|, log-odds)


Scheduled departure block    0.3982
Month                        0.2049
Carrier                      0.1344
Day of week                  0.1054
Destination airport          0.0855
Origin airport               0.0768
Route distance               0.0476
Prior-month load factor      0.0391
dtype: float32

In [10]:
# Figure 13: ROC curves
fig, ax = plt.subplots(figsize=(4.8, 4.2))
for (name, p), col in zip([("Logistic regression", p_lr), ("Random forest", p_rf), ("XGBoost", p_xg)], PALETTE):
    fpr, tpr, _ = roc_curve(y_te, p); ax.plot(fpr, tpr, color=col, label=f"{name} (AUC {roc_auc_score(y_te, p):.3f})")
ax.plot([0, 1], [0, 1], color=INK_2, ls="--", lw=1, label="Baseline (AUC 0.500)")
ax.set_xlabel("False-positive rate"); ax.set_ylabel("True-positive rate"); ax.legend(loc="lower right", fontsize=8)
fig.tight_layout(); save_fig(fig, "fig13_rq1_roc")

# Figure 14: factor influence - two views of the same question side by side
order = shap_imp.index.tolist()[::-1]
fig, ax = plt.subplots(1, 2, figsize=(6.5, 3.4), sharey=True)
ax[0].barh(order, [shap_imp[o] for o in order], color=PALETTE[0], height=0.65); ax[0].set_xlabel("Extreme gradient boosting:\nmean absolute Shapley contribution")
lr_vals = ftest.set_index("Factor")["LR_chi2"]
ax[1].barh(order, [lr_vals[o] for o in order], color=PALETTE[1], height=0.65); ax[1].set_xlabel("Logistic regression:\nlikelihood-ratio chi-square")
for a, vals in zip(ax, ([shap_imp[o] for o in order], [lr_vals[o] for o in order])):
    a.set_xlim(0, max(vals) * 1.2)
fig.tight_layout(); save_fig(fig, "fig14_rq1_importance")

# Figure 15: odds ratios of the strongest individual terms (readable labels, reference level stated)
from matplotlib.ticker import ScalarFormatter
MON = MONTH_NAMES
DOW = {str(i + 1): n for i, n in enumerate(DOW_NAMES)}
def nice(term):
    if term == "distance_k": return "Distance (per additional 1,000 miles)"
    if term == "lf_prev_10": return "Prior-month load factor (per additional 10 percentage points)"
    if term == "lf_prev_missing": return "Prior-month load factor unavailable"
    pre, lvl = term.split("_", 1)
    if pre == "month": return f"Month: {MON[int(lvl)-1]}"
    if pre == "dow": return f"Weekday: {DOW.get(lvl, lvl)}"
    if pre == "blk": return f"Scheduled departure {lvl[:2]}:00 block"
    return {"carr": "Carrier", "orig": "Origin airport", "dest": "Destination airport"}[pre] + ": " + (carrier_name(lvl) if pre == "carr" else airport_name(lvl) if pre in ("orig", "dest") else lvl)
sel = coef.drop(index="const").assign(abs_z=lambda d: d["z"].abs()).sort_values("abs_z", ascending=False).head(14).iloc[::-1].copy()
sel["label"] = [nice(t) for t in sel.index]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.errorbar(sel["OR"], range(len(sel)), xerr=[sel["OR"] - sel["OR_lo"], sel["OR_hi"] - sel["OR"]], fmt="o", color=PALETTE[0], capsize=2.5)
ax.axvline(1, color=INK_2, ls="--", lw=1); ax.set_xscale("log")
lo, hi = float(sel["OR_lo"].min()), float(sel["OR_hi"].max())
ticks = [t for t in (0.2, 0.3, 0.5, 0.7, 1, 1.5, 2, 3) if lo * 0.9 <= t <= hi * 1.1]
ax.set_xticks(ticks); ax.xaxis.set_major_formatter(ScalarFormatter()); ax.minorticks_off()
ax.set_yticks(range(len(sel))); ax.set_yticklabels(sel["label"], fontsize=8)
ax.set_xlabel("Odds ratio of arriving 15 or more minutes late\n(logarithmic scale, 95% confidence interval; 1 = no effect)")
fig.tight_layout(); save_fig(fig, "fig15_rq1_odds")

'outputs/figures/fig15_rq1_odds.png'

In [11]:
base_rate = float(np.mean(y))
rq1 = {
    "n_total": int(len(df)), "n_train": int(len(y_tr)), "n_test": int(len(y_te)), "prevalence": base_rate, "threshold": prev,
    "confusion_lr": conf_lr, "temporal": temporal, "design_columns": int(X.shape[1]), "reference_levels": refs, "overall_lr_test": overall,
    "factor_tests": ftest.round(6).to_dict("records"), "model_comparison": comp.round(5).reset_index().to_dict("records"),
    "importance_method": shap_method, "importance": shap_imp.round(5).to_dict(),
    "top_terms": sel.iloc[::-1][["label", "OR", "OR_lo", "OR_hi", "z", "p"]].round(5).to_dict("records"),
    "distance_or_per_1000mi": float(coef.loc["distance_k", "OR"]), "distance_p": float(coef.loc["distance_k", "p"]),
    "lf_prev_or_per_10pts": float(coef.loc["lf_prev_10", "OR"]), "lf_prev_p": float(coef.loc["lf_prev_10", "p"]),
    "lf_prev_missing_share_pct": float(df["lf_prev_missing"].mean() * 100),
    "decision": "reject H0" if overall["p_value"] < 0.05 else "fail to reject H0",
    "meets_min_n": bool(len(df) >= load_results()["sample_size"]["rq1"]),
}
save_result("rq1", rq1)
print(rq1["decision"], "| overall LR chi2 =", round(overall["llr_chi2"], 1), "df =", overall["df"])

reject H0 | overall LR chi2 = 17898.4 df = 102
